# Multi-Label Clothing Classification Pipeline

Pipeline untuk klasifikasi jenis dan warna pakaian dengan object localization untuk mengurangi noise background.

## 1. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision import models
from ultralytics import YOLO

import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Load dan Eksplorasi Data

In [2]:
train_df = pd.read_csv('train.csv')
sample_sub = pd.read_csv('sample_submission.csv')

print(f"Train data shape: {train_df.shape}")
print(f"Test data count: {len(sample_sub)}")
print("\nTrain data distribution:")
print(f"Jenis - Kaos: {(train_df['jenis']==0).sum()}, Hoodie: {(train_df['jenis']==1).sum()}")
print(f"Warna - Merah: {(train_df['warna']==0).sum()}, Kuning: {(train_df['warna']==1).sum()}, Biru: {(train_df['warna']==2).sum()}, Hitam: {(train_df['warna']==3).sum()}, Putih: {(train_df['warna']==4).sum()}")
print("\nSample data:")
print(train_df.head(10))

Train data shape: (777, 3)
Test data count: 334

Train data distribution:
Jenis - Kaos: 476, Hoodie: 301
Warna - Merah: 116, Kuning: 125, Biru: 162, Hitam: 234, Putih: 140

Sample data:
   id  jenis  warna
0   1      1      1
1   2      0      2
2   3      1      3
3   4      1      1
4   5      0      4
5   6      1      3
6   7      0      4
7   8      1      2
8   9      1      3
9  10      1      0


## Calculate Class Weights untuk Handle Imbalance

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Hitung class weights untuk JENIS (kaos vs hoodie)
jenis_counts = train_df['jenis'].value_counts().sort_index()
print("Distribusi JENIS:")
print(f"  Kaos (0): {jenis_counts[0]} samples ({jenis_counts[0]/len(train_df)*100:.1f}%)")
print(f"  Hoodie (1): {jenis_counts[1]} samples ({jenis_counts[1]/len(train_df)*100:.1f}%)")
print(f"  Imbalance ratio: {jenis_counts[0]/jenis_counts[1]:.2f}:1")

jenis_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=train_df['jenis'].values
)
jenis_weights_tensor = torch.FloatTensor(jenis_weights).to(device)
print(f"  Class weights: Kaos={jenis_weights[0]:.4f}, Hoodie={jenis_weights[1]:.4f}")

# Hitung class weights untuk WARNA (5 colors)
warna_counts = train_df['warna'].value_counts().sort_index()
print("\nDistribusi WARNA:")
warna_names = ['Merah', 'Kuning', 'Biru', 'Hitam', 'Putih']
for i, name in enumerate(warna_names):
    count = warna_counts[i]
    print(f"  {name} ({i}): {count} samples ({count/len(train_df)*100:.1f}%)")

max_warna = warna_counts.max()
min_warna = warna_counts.min()
print(f"  Imbalance ratio: {max_warna/min_warna:.2f}:1 (Hitam vs Merah)")

warna_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2, 3, 4]),
    y=train_df['warna'].values
)
warna_weights_tensor = torch.FloatTensor(warna_weights).to(device)
print(f"  Class weights: ", end='')
for i, name in enumerate(warna_names):
    print(f"{name}={warna_weights[i]:.4f}", end=', ' if i < 4 else '\n')


## 3. Object Detection dengan YOLO untuk Segmentasi Pakaian

In [ ]:
def extract_clothing_region(image_path, yolo_model, conf_threshold=0.3):
    img = cv2.imread(image_path)
    
    if img is None:
        raise FileNotFoundError(f"Cannot read image: {image_path}")
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    results = yolo_model(img_rgb, verbose=False)
    
    best_box = None
    best_conf = 0
    
    for result in results:
        boxes = result.boxes
        for box in boxes:
            cls = int(box.cls[0])
            conf = float(box.conf[0])
            
            if cls == 0 and conf > conf_threshold and conf > best_conf:
                best_conf = conf
                best_box = box.xyxy[0].cpu().numpy()
    
    if best_box is not None:
        x1, y1, x2, y2 = map(int, best_box)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(img_rgb.shape[1], x2), min(img_rgb.shape[0], y2)
        
        cropped = img_rgb[y1:y2, x1:x2]
        return Image.fromarray(cropped)
    else:
        return Image.fromarray(img_rgb)

yolo_model = YOLO('yolov8n.pt')
print("YOLO model loaded")

## 4. Custom Dataset untuk Multi-Label Classification

In [ ]:
class ClothingDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, yolo_model=None, use_yolo=True, is_test=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.yolo_model = yolo_model
        self.use_yolo = use_yolo
        self.is_test = is_test
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_id = self.df.iloc[idx]['id']
        
        img_path = None
        for ext in ['.jpg', '.png']:
            path = os.path.join(self.img_dir, f'{img_id}{ext}')
            if os.path.exists(path):
                img_path = path
                break
        
        if img_path is None:
            raise FileNotFoundError(f"Image not found for id {img_id} in {self.img_dir}")
        
        if self.use_yolo and self.yolo_model is not None:
            image = extract_clothing_region(img_path, self.yolo_model)
        else:
            image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        if self.is_test:
            return image, img_id
        else:
            jenis = self.df.iloc[idx]['jenis']
            warna = self.df.iloc[idx]['warna']
            return image, torch.tensor(jenis, dtype=torch.long), torch.tensor(warna, dtype=torch.long)

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.25, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def collate_fn(batch):
    if isinstance(batch[0], tuple) and len(batch[0]) == 3:
        images = torch.stack([item[0] for item in batch])
        jenis = torch.stack([item[1] for item in batch])
        warna = torch.stack([item[2] for item in batch])
        return images, jenis, warna
    elif isinstance(batch[0], tuple) and len(batch[0]) == 2:
        images = torch.stack([item[0] for item in batch])
        ids = [item[1] for item in batch]
        return images, ids
    else:
        raise ValueError(f"Unexpected batch structure: {type(batch[0])}, length: {len(batch[0]) if isinstance(batch[0], tuple) else 'N/A'}")

## 5. Model Multi-Label Classification

In [ ]:
class MultiLabelClothingModel(nn.Module):
    def __init__(self, num_jenis=2, num_warna=5):
        super(MultiLabelClothingModel, self).__init__()
        
        self.backbone = models.efficientnet_b0(pretrained=True)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()
        
        self.jenis_head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_jenis)
        )
        
        self.warna_head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_warna)
        )
    
    def forward(self, x):
        features = self.backbone(x)
        jenis_out = self.jenis_head(features)
        warna_out = self.warna_head(features)
        return jenis_out, warna_out

model = MultiLabelClothingModel().to(device)
print("Model initialized")

## 5b. Model dengan Spatial Attention (Improved)

In [ ]:
class SpatialAttention(nn.Module):
    """
    Spatial Attention Module - Fokus ke region penting (kerah, tudung, area pakaian)
    """
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        # x: (B, C, H, W)
        avg_out = torch.mean(x, dim=1, keepdim=True)  # (B, 1, H, W)
        max_out, _ = torch.max(x, dim=1, keepdim=True)  # (B, 1, H, W)
        
        # Concat channel-wise statistics
        concat = torch.cat([avg_out, max_out], dim=1)  # (B, 2, H, W)
        
        # Generate attention map
        attention = self.sigmoid(self.conv(concat))  # (B, 1, H, W)
        
        # Apply attention
        return x * attention

class ChannelAttention(nn.Module):
    """
    Channel Attention Module - Fokus ke feature channels yang penting
    """
    def __init__(self, in_channels, reduction=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(),
            nn.Linear(in_channels // reduction, in_channels, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        # x: (B, C, H, W)
        b, c, _, _ = x.size()
        
        # Global pooling
        avg_out = self.fc(self.avg_pool(x).view(b, c))
        max_out = self.fc(self.max_pool(x).view(b, c))
        
        # Attention weights
        attention = self.sigmoid(avg_out + max_out).view(b, c, 1, 1)
        
        # Apply attention
        return x * attention

class AttentionClothingModel(nn.Module):
    """
    Model dengan Spatial + Channel Attention untuk fokus ke kerah/tudung dan warna
    """
    def __init__(self, num_jenis=2, num_warna=5):
        super(AttentionClothingModel, self).__init__()
        
        # Backbone
        self.backbone = models.efficientnet_b0(pretrained=True)
        in_features = self.backbone.classifier[1].in_features
        
        # Extract feature extractor (tanpa classifier)
        self.features = self.backbone.features
        
        # Attention modules setelah backbone
        self.channel_attention = ChannelAttention(in_features)
        self.spatial_attention = SpatialAttention(kernel_size=7)
        
        # Global pooling
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        
        # Separate heads dengan attention
        self.jenis_head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_jenis)
        )
        
        self.warna_head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_warna)
        )
    
    def forward(self, x):
        # Extract features
        x = self.features(x)  # (B, C, H, W)
        
        # Apply attention mechanisms
        x = self.channel_attention(x)  # Fokus ke feature channels penting
        x = self.spatial_attention(x)  # Fokus ke spatial regions (kerah/tudung/pakaian)
        
        # Global pooling
        x = self.avgpool(x)  # (B, C, 1, 1)
        x = torch.flatten(x, 1)  # (B, C)
        
        # Classification heads
        jenis_out = self.jenis_head(x)
        warna_out = self.warna_head(x)
        
        return jenis_out, warna_out

# PILIH MODEL: Original atau Attention
USE_ATTENTION = True  # Set True untuk gunakan attention model

if USE_ATTENTION:
    model = AttentionClothingModel(num_jenis=2, num_warna=5).to(device)
    print("Model dengan Spatial + Channel Attention initialized")
else:
    model = MultiLabelClothingModel(num_jenis=2, num_warna=5).to(device)
    print("Model baseline initialized")

# Print model info
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 6. Training Setup

In [ ]:
train_data, val_data = train_test_split(train_df, test_size=0.15, random_state=42, stratify=train_df['jenis'])

train_dataset = ClothingDataset(train_data, 'train/train', train_transform, yolo_model, use_yolo=True)
val_dataset = ClothingDataset(val_data, 'train/train', test_transform, yolo_model, use_yolo=True)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0, collate_fn=collate_fn)

# Loss functions dengan class weights untuk handle imbalance
criterion_jenis = nn.CrossEntropyLoss(weight=jenis_weights_tensor)
criterion_warna = nn.CrossEntropyLoss(weight=warna_weights_tensor)

optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)

print(f"Train samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")

## 7. Training Loop

In [ ]:
def train_epoch(model, loader, criterion_jenis, criterion_warna, optimizer, device):
    model.train()
    total_loss = 0
    jenis_correct = 0
    warna_correct = 0
    total = 0
    
    for images, jenis_labels, warna_labels in loader:
        images = images.to(device)
        jenis_labels = jenis_labels.to(device)
        warna_labels = warna_labels.to(device)
        
        optimizer.zero_grad()
        jenis_out, warna_out = model(images)
        
        loss_jenis = criterion_jenis(jenis_out, jenis_labels)
        loss_warna = criterion_warna(warna_out, warna_labels)
        loss = loss_jenis + loss_warna
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        jenis_correct += (jenis_out.argmax(1) == jenis_labels).sum().item()
        warna_correct += (warna_out.argmax(1) == warna_labels).sum().item()
        total += jenis_labels.size(0)
    
    return total_loss / len(loader), jenis_correct / total, warna_correct / total

def validate(model, loader, criterion_jenis, criterion_warna, device):
    model.eval()
    total_loss = 0
    jenis_correct = 0
    warna_correct = 0
    total = 0
    
    with torch.no_grad():
        for images, jenis_labels, warna_labels in loader:
            images = images.to(device)
            jenis_labels = jenis_labels.to(device)
            warna_labels = warna_labels.to(device)
            
            jenis_out, warna_out = model(images)
            
            loss_jenis = criterion_jenis(jenis_out, jenis_labels)
            loss_warna = criterion_warna(warna_out, warna_labels)
            loss = loss_jenis + loss_warna
            
            total_loss += loss.item()
            jenis_correct += (jenis_out.argmax(1) == jenis_labels).sum().item()
            warna_correct += (warna_out.argmax(1) == warna_labels).sum().item()
            total += jenis_labels.size(0)
    
    return total_loss / len(loader), jenis_correct / total, warna_correct / total

## Evaluasi dengan Classification Report

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize
import seaborn as sns
import matplotlib.pyplot as plt

def evaluate_model(model, loader, device):
    """
    Evaluasi model dengan classification report, F1-score, dan ROC-AUC untuk setiap task
    """
    model.eval()
    
    all_jenis_preds = []
    all_jenis_labels = []
    all_jenis_probs = []  # Untuk ROC-AUC
    all_warna_preds = []
    all_warna_labels = []
    all_warna_probs = []  # Untuk ROC-AUC
    
    with torch.no_grad():
        for images, jenis_labels, warna_labels in loader:
            images = images.to(device)
            jenis_out, warna_out = model(images)
            
            # Predictions
            jenis_preds = jenis_out.argmax(1).cpu().numpy()
            warna_preds = warna_out.argmax(1).cpu().numpy()
            
            # Probabilities (untuk ROC-AUC)
            jenis_probs = torch.softmax(jenis_out, dim=1).cpu().numpy()
            warna_probs = torch.softmax(warna_out, dim=1).cpu().numpy()
            
            all_jenis_preds.extend(jenis_preds)
            all_jenis_labels.extend(jenis_labels.numpy())
            all_jenis_probs.extend(jenis_probs)
            all_warna_preds.extend(warna_preds)
            all_warna_labels.extend(warna_labels.numpy())
            all_warna_probs.extend(warna_probs)
    
    # Convert to numpy arrays
    all_jenis_probs = np.array(all_jenis_probs)
    all_warna_probs = np.array(all_warna_probs)
    
    # ========== JENIS (Kaos vs Hoodie) ==========
    print("="*60)
    print("CLASSIFICATION REPORT - JENIS (Kaos vs Hoodie)")
    print("="*60)
    jenis_names = ['kaos', 'hoodie']
    print(classification_report(all_jenis_labels, all_jenis_preds, 
                                target_names=jenis_names, digits=4))
    
    # Micro F1-Score untuk JENIS
    jenis_micro_f1 = f1_score(all_jenis_labels, all_jenis_preds, average='micro')
    jenis_macro_f1 = f1_score(all_jenis_labels, all_jenis_preds, average='macro')
    print(f"\nF1-Scores:")
    print(f"   Micro F1-Score: {jenis_micro_f1:.4f}")
    print(f"   Macro F1-Score: {jenis_macro_f1:.4f}")
    
    # ROC-AUC untuk JENIS (binary classification)
    jenis_auc = roc_auc_score(all_jenis_labels, all_jenis_probs[:, 1])
    print(f"\nROC-AUC Score: {jenis_auc:.4f}")
    
    # Confusion Matrix untuk JENIS
    cm_jenis = confusion_matrix(all_jenis_labels, all_jenis_preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm_jenis, annot=True, fmt='d', cmap='Blues',
                xticklabels=jenis_names, yticklabels=jenis_names)
    plt.title(f'Confusion Matrix - Jenis\nMicro F1: {jenis_micro_f1:.4f} | AUC: {jenis_auc:.4f}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()
    
    # ========== WARNA (5 Colors) ==========
    print("\n" + "="*60)
    print("CLASSIFICATION REPORT - WARNA (5 Colors)")
    print("="*60)
    warna_names = ['merah', 'kuning', 'biru', 'hitam', 'putih']
    print(classification_report(all_warna_labels, all_warna_preds,
                                target_names=warna_names, digits=4))
    
    # Micro F1-Score untuk WARNA
    warna_micro_f1 = f1_score(all_warna_labels, all_warna_preds, average='micro')
    warna_macro_f1 = f1_score(all_warna_labels, all_warna_preds, average='macro')
    print(f"\nF1-Scores:")
    print(f"   Micro F1-Score: {warna_micro_f1:.4f}")
    print(f"   Macro F1-Score: {warna_macro_f1:.4f}")
    
    # ROC-AUC untuk WARNA (multiclass - one-vs-rest)
    warna_auc = roc_auc_score(all_warna_labels, all_warna_probs, multi_class='ovr', average='macro')
    print(f"\nROC-AUC Score (macro): {warna_auc:.4f}")
    
    # Confusion Matrix untuk WARNA
    cm_warna = confusion_matrix(all_warna_labels, all_warna_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_warna, annot=True, fmt='d', cmap='Greens',
                xticklabels=warna_names, yticklabels=warna_names)
    plt.title(f'Confusion Matrix - Warna\nMicro F1: {warna_micro_f1:.4f} | AUC: {warna_auc:.4f}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()
    
    return {
        'jenis': {
            'predictions': all_jenis_preds,
            'labels': all_jenis_labels,
            'probabilities': all_jenis_probs,
            'confusion_matrix': cm_jenis,
            'micro_f1': jenis_micro_f1,
            'macro_f1': jenis_macro_f1,
            'auc': jenis_auc
        },
        'warna': {
            'predictions': all_warna_preds,
            'labels': all_warna_labels,
            'probabilities': all_warna_probs,
            'confusion_matrix': cm_warna,
            'micro_f1': warna_micro_f1,
            'macro_f1': warna_macro_f1,
            'auc': warna_auc
        }
    }
    
# Jalankan setelah training selesai
# results = evaluate_model(model, val_loader, device)

### Jalankan Evaluasi pada Validation Set

Setelah training selesai, panggil fungsi ini untuk melihat performa detail per kelas:

In [ ]:
# Jalankan setelah training selesai
# results = evaluate_model(model, val_loader, device)

## Visualisasi ROC Curves

In [ ]:
def plot_roc_curves(results):
    """
    Plot ROC curves untuk JENIS dan WARNA
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # ========== ROC Curve untuk JENIS ==========
    jenis_labels = results['jenis']['labels']
    jenis_probs = results['jenis']['probabilities']
    
    # ROC untuk binary classification (Kaos vs Hoodie)
    fpr, tpr, _ = roc_curve(jenis_labels, jenis_probs[:, 1])
    auc_score = results['jenis']['auc']
    
    axes[0].plot(fpr, tpr, color='darkorange', lw=2, 
                 label=f'ROC curve (AUC = {auc_score:.4f})')
    axes[0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
    axes[0].set_xlim([0.0, 1.0])
    axes[0].set_ylim([0.0, 1.05])
    axes[0].set_xlabel('False Positive Rate', fontsize=12)
    axes[0].set_ylabel('True Positive Rate', fontsize=12)
    axes[0].set_title('ROC Curve - JENIS (Kaos vs Hoodie)', fontsize=14, fontweight='bold')
    axes[0].legend(loc="lower right", fontsize=11)
    axes[0].grid(True, alpha=0.3)
    
    # ========== ROC Curve untuk WARNA (One-vs-Rest) ==========
    warna_labels = results['warna']['labels']
    warna_probs = results['warna']['probabilities']
    warna_names = ['Merah', 'Kuning', 'Biru', 'Hitam', 'Putih']
    colors = ['red', 'gold', 'blue', 'black', 'lightcoral']
    
    # Binarize labels untuk multiclass ROC
    warna_labels_bin = label_binarize(warna_labels, classes=[0, 1, 2, 3, 4])
    
    # Plot ROC untuk setiap kelas warna
    for i, (color, name) in enumerate(zip(colors, warna_names)):
        fpr, tpr, _ = roc_curve(warna_labels_bin[:, i], warna_probs[:, i])
        auc_score = roc_auc_score(warna_labels_bin[:, i], warna_probs[:, i])
        axes[1].plot(fpr, tpr, color=color, lw=2, 
                     label=f'{name} (AUC = {auc_score:.3f})')
    
    # Plot random classifier line
    axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
    
    axes[1].set_xlim([0.0, 1.0])
    axes[1].set_ylim([0.0, 1.05])
    axes[1].set_xlabel('False Positive Rate', fontsize=12)
    axes[1].set_ylabel('True Positive Rate', fontsize=12)
    axes[1].set_title(f'ROC Curves - WARNA (One-vs-Rest)\nMacro AUC = {results["warna"]["auc"]:.4f}', 
                     fontsize=14, fontweight='bold')
    axes[1].legend(loc="lower right", fontsize=10)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
# plot_roc_curves(results)

## 8. Train Model

In [ ]:
num_epochs = 20
best_val_loss = float('inf')
best_model_path = 'best_clothing_model.pth'

history = {'train_loss': [], 'val_loss': [], 'train_jenis_acc': [], 'val_jenis_acc': [], 'train_warna_acc': [], 'val_warna_acc': []}

for epoch in range(num_epochs):
    train_loss, train_jenis_acc, train_warna_acc = train_epoch(model, train_loader, criterion_jenis, criterion_warna, optimizer, device)
    val_loss, val_jenis_acc, val_warna_acc = validate(model, val_loader, criterion_jenis, criterion_warna, device)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_jenis_acc'].append(train_jenis_acc)
    history['val_jenis_acc'].append(val_jenis_acc)
    history['train_warna_acc'].append(train_warna_acc)
    history['val_warna_acc'].append(val_warna_acc)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}, Jenis Acc: {train_jenis_acc:.4f}, Warna Acc: {train_warna_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Jenis Acc: {val_jenis_acc:.4f}, Warna Acc: {val_warna_acc:.4f}")
    
    scheduler.step(val_loss)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        print(f"Model saved with val_loss: {val_loss:.4f}")
    print()

print("Training completed")

## 9. Visualisasi Training History

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss over Epochs')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history['train_jenis_acc'], label='Train Jenis Acc')
axes[1].plot(history['val_jenis_acc'], label='Val Jenis Acc')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Jenis Accuracy over Epochs')
axes[1].legend()
axes[1].grid(True)

axes[2].plot(history['train_warna_acc'], label='Train Warna Acc')
axes[2].plot(history['val_warna_acc'], label='Val Warna Acc')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Accuracy')
axes[2].set_title('Warna Accuracy over Epochs')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

## 10. Load Best Model dan Prediksi Test Data

In [ ]:
model.load_state_dict(torch.load(best_model_path))
model.eval()
print("Best model loaded")

test_dataset = ClothingDataset(sample_sub, 'test/test', test_transform, yolo_model, use_yolo=True, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0, collate_fn=collate_fn)

predictions_jenis = []
predictions_warna = []
test_ids = []

with torch.no_grad():
    for images, img_ids in test_loader:
        images = images.to(device)
        jenis_out, warna_out = model(images)
        
        jenis_pred = jenis_out.argmax(1).cpu().numpy()
        warna_pred = warna_out.argmax(1).cpu().numpy()
        
        predictions_jenis.extend(jenis_pred)
        predictions_warna.extend(warna_pred)
        test_ids.extend(img_ids)

print(f"Predictions completed for {len(test_ids)} test images")

## 11. Create Submission File

In [ ]:
submission = pd.DataFrame({
    'id': test_ids,
    'jenis': predictions_jenis,
    'warna': predictions_warna
})

submission = submission.sort_values('id').reset_index(drop=True)

submission.to_csv('submission.csv', index=False)
print("Submission file created: submission.csv")
print(f"\nSubmission shape: {submission.shape}")
print("\nFirst 10 predictions:")
print(submission.head(10))
print("\nPrediction distribution:")
print(f"Jenis - Kaos: {(submission['jenis']==0).sum()}, Hoodie: {(submission['jenis']==1).sum()}")
print(f"Warna - Merah: {(submission['warna']==0).sum()}, Kuning: {(submission['warna']==1).sum()}, Biru: {(submission['warna']==2).sum()}, Hitam: {(submission['warna']==3).sum()}, Putih: {(submission['warna']==4).sum()}")

## 12. Visualisasi Sample Predictions

In [ ]:
jenis_map = {0: 'Kaos', 1: 'Hoodie'}
warna_map = {0: 'Merah', 1: 'Kuning', 2: 'Biru', 3: 'Hitam', 4: 'Putih'}

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i in range(8):
    idx = np.random.randint(0, len(test_ids))
    img_id = test_ids[idx]
    
    img_path = None
    for ext in ['.jpg', '.png']:
        path = os.path.join('test/test', f'{img_id}{ext}')
        if os.path.exists(path):
            img_path = path
            break
    
    img = Image.open(img_path).convert('RGB')
    
    pred_jenis = predictions_jenis[idx]
    pred_warna = predictions_warna[idx]
    
    axes[i].imshow(img)
    axes[i].axis('off')
    axes[i].set_title(f'ID: {img_id}\n{jenis_map[pred_jenis]} - {warna_map[pred_warna]}', fontsize=10)

plt.tight_layout()
plt.show()